# 02 — Feature Engineering (v2)

**Objetivo:** Transformar el dataset limpio en un dataset listo para modelado.

**Transformaciones:**
1. Eliminar features sin valor predictivo (`anio_creacion`, `subtipo_interes`, `plataforma` — data leakage)
2. Crear features derivadas (`es_fin_de_semana`, `franja_horaria`)
3. Agrupar categorías de baja frecuencia en "otros"
4. **Encoding inteligente de variables categóricas:**
   - **Contribución + Bayesian smoothing** para: `vehiculo_interes`, `origen`, `nombre_formulario`, `campana` (corrige sesgo de volumen)
   - **Target encoding** para: `concesionario` (alta cardinalidad)
   - **One-hot encoding** para: `dia_semana_creacion`, `franja_horaria`, `origen_creacion` (baja cardinalidad)
5. Split train/test estratificado
6. Exportar datasets finales

**Cambio clave vs v1:** El one-hot encoding de vehículo/origen/formulario/campaña causaba que KWID (el más vendido) fuera clasificado como Cold, porque categorías de bajo volumen (LOGAN n=2) inflaban el promedio con tasas del 100%. El nuevo encoding de 2 features por variable corrige esto.

## 1. Cargar dataset limpio

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

CLEAN_PATH = "../data/processed/leads_cleaned.csv"
df = pd.read_csv(CLEAN_PATH)

print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Columnas: {list(df.columns)}")

Dataset cargado: 8,422 filas x 14 columnas
Columnas: ['anio_creacion', 'mes_creacion', 'dia_creacion', 'hora_creacion', 'dia_semana_creacion', 'nombre_formulario', 'campana', 'plataforma', 'origen_creacion', 'subtipo_interes', 'vehiculo_interes', 'concesionario', 'origen', 'target']


## 2. Eliminar features sin valor predictivo

Según el EDA:
- **`anio_creacion`**: Solo 2 valores (2025/2026) con poca variabilidad.
- **`subtipo_interes`**: El 96.5% es "Solicitud de compra". No discrimina.
- **`plataforma`**: Solo 2 valores. `MX_LEAD_QUALIF` indica leads ya clasificados como Hot → **data leakage** (concentraba 41.2% de importancia del modelo v1).

In [2]:
cols_eliminar = ["anio_creacion", "subtipo_interes", "plataforma"]

print("Features eliminadas:")
for col in cols_eliminar:
    print(f"  - {col}: {df[col].nunique()} valores únicos")
    print(f"    Distribución: {df[col].value_counts().head(3).to_dict()}")

df = df.drop(columns=cols_eliminar)
print(f"\nDataset: {df.shape[0]:,} filas x {df.shape[1]} columnas")

Features eliminadas:
  - anio_creacion: 2 valores únicos
    Distribución: {2025: 6358, 2026: 2064}
  - subtipo_interes: 5 valores únicos
    Distribución: {'Solicitud de compra': 8129, 'Solicitud general': 208, 'TestDrive Request': 76}
  - plataforma: 2 valores únicos
    Distribución: {'MX_LEAD_CHATBOT_QUALIF': 5111, 'MX_LEAD_QUALIF': 3311}

Dataset: 8,422 filas x 11 columnas


## 3. Crear nuevas features derivadas

Basado en los hallazgos del EDA:
- **`es_fin_de_semana`**: Binaria (1 = sábado/domingo, 0 = lun-vie). El EDA mostró que los leads de fin de semana convierten más.
- **`franja_horaria`**: Agrupa las 24 horas en 4 franjas. El EDA mostró que la madrugada convierte significativamente más.

In [3]:
df["es_fin_de_semana"] = df["dia_semana_creacion"].isin(["sábado", "domingo"]).astype(int)

def clasificar_franja(hora):
    if 0 <= hora < 6:
        return "madrugada"
    elif 6 <= hora < 12:
        return "manana"
    elif 12 <= hora < 18:
        return "tarde"
    else:
        return "noche"

df["franja_horaria"] = df["hora_creacion"].apply(clasificar_franja)

print("Nuevas features creadas:\n")
print("es_fin_de_semana:")
print(df["es_fin_de_semana"].value_counts().to_string())
print(f"\nfranja_horaria:")
print(df["franja_horaria"].value_counts().to_string())

print(f"\nConversión por franja horaria:")
for franja in ["madrugada", "manana", "tarde", "noche"]:
    subset = df[df["franja_horaria"]==franja]
    print(f"  {franja:12s} → {subset['target'].mean()*100:.1f}% Hot ({len(subset):,} leads)")

Nuevas features creadas:

es_fin_de_semana:
es_fin_de_semana
0    5339
1    3083

franja_horaria:
franja_horaria
tarde        2899
noche        2732
manana       1506
madrugada    1285

Conversión por franja horaria:
  madrugada    → 72.2% Hot (1,285 leads)
  manana       → 68.0% Hot (1,506 leads)
  tarde        → 66.3% Hot (2,899 leads)
  noche        → 69.8% Hot (2,732 leads)


## 4. Agrupar categorías de baja frecuencia

Las categorías con menos del 1% del total se agrupan en "otros" para reducir ruido y dimensionalidad.

In [4]:
UMBRAL_PCT = 1.0
umbral_abs = int(len(df) * UMBRAL_PCT / 100)

cat_cols_agrupar = ["nombre_formulario", "vehiculo_interes", "origen", "campana", "concesionario"]

print(f"Umbral: {UMBRAL_PCT}% = {umbral_abs} leads\n")

for col in cat_cols_agrupar:
    vc = df[col].value_counts()
    bajas = vc[vc < umbral_abs].index
    if len(bajas) > 0:
        n_antes = df[col].nunique()
        df.loc[df[col].isin(bajas), col] = "otros"
        n_despues = df[col].nunique()
        print(f"  {col}: {n_antes} → {n_despues} categorías ({len(bajas)} agrupadas en 'otros')")
    else:
        print(f"  {col}: sin cambios ({df[col].nunique()} categorías, todas >= umbral)")

Umbral: 1.0% = 84 leads

  nombre_formulario: 15 → 9 categorías (7 agrupadas en 'otros')
  vehiculo_interes: 16 → 9 categorías (8 agrupadas en 'otros')
  origen: 16 → 9 categorías (8 agrupadas en 'otros')
  campana: 23 → 9 categorías (15 agrupadas en 'otros')
  concesionario: 93 → 42 categorías (52 agrupadas en 'otros')


## 5. Encoding de variables categóricas

### Problema detectado: sesgo de volumen en one-hot encoding

Categorías de bajo volumen (LOGAN n=2, campañas con n=1) tienen tasas del 100%, mientras que las de alto volumen (KWID n=3,243) tienen tasas más bajas pero aportan la mayoría de Hot Leads. Con one-hot, el modelo aprendía KWID → Cold (53.7% < global 69%).

### Solución: 2 features por variable

Para `vehiculo_interes`, `origen`, `nombre_formulario`, `campana`:
1. **Contribución Hot** = `hot_categoria / total_hot` → ¿Cuánto aporta al negocio?
2. **Tasa suavizada** = `(n × tasa + m × global) / (n + m)` con m=100 → Conversión protegida contra muestras pequeñas

**Encoding restante:**
- **Target encoding**: `concesionario`
- **One-hot**: `dia_semana_creacion`, `franja_horaria`, `origen_creacion` (baja cardinalidad)

**Todo se calcula SOLO con train.**

### 5.1 Split train/test (antes del encoding)

Se divide el dataset **antes** de aplicar el encoding para evitar **data leakage**. Si calculáramos el target encoding con todo el dataset, el modelo vería información del test durante el entrenamiento, inflando artificialmente las métricas.

- **80% train / 20% test** con `stratify=y` para mantener la proporción de Hot/Cold en ambos conjuntos.

In [5]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]:,} filas ({y_train.mean()*100:.1f}% Hot)")
print(f"Test:  {X_test.shape[0]:,} filas ({y_test.mean()*100:.1f}% Hot)")
print(f"\nStratificación verificada: proporciones similares en train y test")

Train: 6,737 filas (68.7% Hot)
Test:  1,685 filas (68.7% Hot)

Stratificación verificada: proporciones similares en train y test


### 5.2 Target encoding para `concesionario`

`concesionario` tiene 42 categorías (después de agrupar). En su lugar de one-hot, se reemplaza cada concesionario por su **tasa de conversión promedio**.

- Se calcula **solo con datos de train** para evitar data leakage.
- Concesionarios desconocidos → **media global** como fallback.

In [6]:
target_enc_col = "concesionario"

train_temp = X_train.copy()
train_temp["target"] = y_train.values
concesionario_means = train_temp.groupby(target_enc_col)["target"].mean()
global_mean = y_train.mean()

X_train["concesionario_target_enc"] = X_train[target_enc_col].map(concesionario_means).fillna(global_mean)
X_test["concesionario_target_enc"] = X_test[target_enc_col].map(concesionario_means).fillna(global_mean)

X_train = X_train.drop(columns=[target_enc_col])
X_test = X_test.drop(columns=[target_enc_col])

print(f"Target encoding aplicado a '{target_enc_col}':")
print(f"  Valores únicos mapeados: {len(concesionario_means)}")
print(f"  Rango: {concesionario_means.min():.3f} - {concesionario_means.max():.3f}")
print(f"  Media global (fallback): {global_mean:.3f}")

Target encoding aplicado a 'concesionario':
  Valores únicos mapeados: 42
  Rango: 0.515 - 0.809
  Media global (fallback): 0.687


### 5.3 Encoding de contribución + Bayesian smoothing

Para las 4 variables con sesgo de volumen → **2 features por variable** (solo con train):
- **`_contribucion_hot`**: Proporción de Hot Leads que aporta esa categoría
- **`_tasa_suavizada`**: Tasa de conversión con prior bayesiano (m=100)

Las restantes (`dia_semana_creacion`, `franja_horaria`, `origen_creacion`) se codifican con one-hot.

In [7]:
# --- Encoding de contribución + Bayesian smoothing ---
SMOOTH_M = 100

encode_2feat_cols = ["vehiculo_interes", "origen", "nombre_formulario", "campana"]

train_temp = X_train.copy()
train_temp["target"] = y_train.values
total_hot_train = y_train.sum()
global_rate = y_train.mean()

print(f"Factor de suavizado (m): {SMOOTH_M}")
print(f"Tasa global: {global_rate:.3f}")
print(f"Total Hot en train: {total_hot_train:,}\n")

encoding_maps = {}

for col in encode_2feat_cols:
    stats = train_temp.groupby(col)["target"].agg(["sum", "count", "mean"])
    stats.columns = ["n_hot", "n_total", "rate"]

    stats["contribucion_hot"] = stats["n_hot"] / total_hot_train
    stats["tasa_suavizada"] = (
        (stats["n_total"] * stats["rate"] + SMOOTH_M * global_rate) /
        (stats["n_total"] + SMOOTH_M)
    )

    encoding_maps[col] = {
        "contribucion": stats["contribucion_hot"].to_dict(),
        "suavizada": stats["tasa_suavizada"].to_dict(),
    }

    contrib_col = f"{col}_contribucion_hot"
    suaviz_col = f"{col}_tasa_suavizada"

    X_train[contrib_col] = X_train[col].map(stats["contribucion_hot"]).fillna(0)
    X_train[suaviz_col] = X_train[col].map(stats["tasa_suavizada"]).fillna(global_rate)
    X_test[contrib_col] = X_test[col].map(stats["contribucion_hot"]).fillna(0)
    X_test[suaviz_col] = X_test[col].map(stats["tasa_suavizada"]).fillna(global_rate)

    X_train = X_train.drop(columns=[col])
    X_test = X_test.drop(columns=[col])

    print(f"✅ {col}:")
    top3 = stats.sort_values("contribucion_hot", ascending=False).head(3)
    for cat, row in top3.iterrows():
        print(f"   {cat:45s} contrib={row['contribucion_hot']:.3f}  suav={row['tasa_suavizada']:.3f}  (n={int(row['n_total']):,}, raw={row['rate']:.1%})")
    print()

# --- One-hot para variables restantes ---
onehot_cols = list(X_train.select_dtypes(include="object").columns)
print(f"One-hot encoding para {len(onehot_cols)} variables restantes:")
for col in onehot_cols:
    print(f"  - {col}: {X_train[col].nunique()} categorías")

X_train = pd.get_dummies(X_train, columns=onehot_cols, drop_first=True, dtype=int)
X_test = pd.get_dummies(X_test, columns=onehot_cols, drop_first=True, dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f"\nDimensiones finales:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")

Factor de suavizado (m): 100
Tasa global: 0.687
Total Hot en train: 4,625

✅ vehiculo_interes:
   KWID                                          contrib=0.485  suav=0.539  (n=4,187, raw=53.6%)
   KOLEOS 2026                                   contrib=0.177  suav=0.962  (n=824, raw=99.5%)
   OROCH                                         contrib=0.094  suav=0.826  (n=510, raw=85.3%)

✅ origen:
   Social-paid                                   contrib=0.476  suav=0.980  (n=2,218, raw=99.3%)
   paid_search                                   contrib=0.257  suav=0.546  (n=2,203, raw=54.0%)
   Organic_search                                contrib=0.117  suav=0.518  (n=1,077, raw=50.2%)

✅ nombre_formulario:
   ONE-PR                                        contrib=0.482  suav=0.524  (n=4,283, raw=52.0%)
   MX_Renault_2025_Kwid_Lead                     contrib=0.275  suav=0.969  (n=1,282, raw=99.1%)
   MX_Renault_2026_Kwid_Lead                     contrib=0.073  suav=0.924  (n=341, raw=99.4%)

✅ ca

### 5.4 Nota sobre data leakage (`plataforma`)

> `plataforma` fue eliminada en el **paso 2** porque contenía data leakage. Ya no requiere acción aquí.

In [8]:
## 6. Verificación final
print("=== VERIFICACIÓN FINAL ===\n")
print(f"X_train: {X_train.shape[0]:,} filas x {X_train.shape[1]} features")
print(f"X_test:  {X_test.shape[0]:,} filas x {X_test.shape[1]} features")
print(f"y_train: {len(y_train):,} ({y_train.mean()*100:.1f}% Hot)")
print(f"y_test:  {len(y_test):,} ({y_test.mean()*100:.1f}% Hot)")

print(f"\nNulos en X_train: {X_train.isnull().sum().sum()}")
print(f"Nulos en X_test:  {X_test.isnull().sum().sum()}")

print(f"\nTipos de datos:")
print(X_train.dtypes.value_counts().to_string())

print(f"\nFeatures finales ({X_train.shape[1]}):")
for col in X_train.columns:
    print(f"  - {col}")

=== VERIFICACIÓN FINAL ===

X_train: 6,737 filas x 24 features
X_test:  1,685 filas x 24 features
y_train: 6,737 (68.7% Hot)
y_test:  1,685 (68.7% Hot)

Nulos en X_train: 0
Nulos en X_test:  0

Tipos de datos:
int64      15
float64     9

Features finales (24):
  - mes_creacion
  - dia_creacion
  - hora_creacion
  - es_fin_de_semana
  - concesionario_target_enc
  - vehiculo_interes_contribucion_hot
  - vehiculo_interes_tasa_suavizada
  - origen_contribucion_hot
  - origen_tasa_suavizada
  - nombre_formulario_contribucion_hot
  - nombre_formulario_tasa_suavizada
  - campana_contribucion_hot
  - campana_tasa_suavizada
  - dia_semana_creacion_jueves
  - dia_semana_creacion_lunes
  - dia_semana_creacion_martes
  - dia_semana_creacion_miércoles
  - dia_semana_creacion_sábado
  - dia_semana_creacion_viernes
  - origen_creacion_ONE
  - origen_creacion_WhatsApp
  - franja_horaria_manana
  - franja_horaria_noche
  - franja_horaria_tarde


## 7. Exportar datasets para modelado

In [9]:
import os

OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

X_train.to_csv(f"{OUTPUT_DIR}/X_train.csv", index=False)
X_test.to_csv(f"{OUTPUT_DIR}/X_test.csv", index=False)
y_train.to_csv(f"{OUTPUT_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{OUTPUT_DIR}/y_test.csv", index=False)

print("Archivos exportados:")
for f in ["X_train.csv", "X_test.csv", "y_train.csv", "y_test.csv"]:
    size = os.path.getsize(f"{OUTPUT_DIR}/{f}")
    print(f"  {f}: {size/1024:.1f} KB")

Archivos exportados:


  X_train.csv: 1298.1 KB
  X_test.csv: 325.4 KB
  y_train.csv: 19.7 KB
  y_test.csv: 4.9 KB


---

**Feature engineering v2 completado.**

**Resumen:**
- Eliminadas: `anio_creacion`, `subtipo_interes`, `plataforma` (data leakage)
- Creadas: `es_fin_de_semana`, `franja_horaria`
- **Encoding contribución + Bayesian (m=100):** `vehiculo_interes`, `origen`, `nombre_formulario`, `campana` → 2 features c/u
- Target encoding: `concesionario` → `concesionario_target_enc`
- One-hot: `dia_semana_creacion`, `franja_horaria`, `origen_creacion`
- Categorías <1% agrupadas en "otros"
- Split 80/20 estratificado